# 03. Feature Engineering: Rácios Financeiros e Governance
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Este notebook prepara e valida os indicadores preditivos (Rácios) a partir dos dados limpos.

**Nota Metodológica (Pipeline):** 
Input: `data/interim/micro_long.csv`  
Output: `data/interim/micro_features.csv`

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')
from src.data.base import safe_float

DATA_INTERIM = "../data/interim"
df = pd.read_csv(os.path.join(DATA_INTERIM, "micro_long.csv"), low_memory=False)

print(f'Dataset carregado: {len(df):,} observações | {df["NIF Code"].nunique():,} empresas')
print(f'Colunas disponíveis: {list(df.columns)}')
print(f'Período: {df["Year"].min()} – {df["Year"].max()}')

Dataset carregado: 127,621 observações | 21,925 empresas
Colunas disponíveis: ['NIF Code', 'Company Name', 'Date of Establishment', 'Status', 'Status Date', 'BoardSize', 'IndependenceIndicator', 'OwnershipConcentration', 'CAE', 'District', 'LegalForm', 'Year', 'Revenue', 'Employees', 'TotalAssets', 'Equity', 'ROA', 'ROE', 'Liquidity', 'Indebtedness', 'NetProfit', 'EBITDA', 'Interests', 'RetainedEarnings', 'PMR', 'PMP', 'AcidTest', 'CashFlow', 'CAE_Sector', 'FirmAge']
Período: 2002 – 2025


## 1. Conversão e Limpeza de Dados SABI

Convertemos as colunas financeiras para formato numérico.

In [2]:
# Ajustado para os nomes reais detetados no ficheiro
fin_cols = ['TotalAssets', 'Equity', 'NetProfit', 'EBITDA', 'Liquidity', 'Indebtedness', 'Revenue', 'Interests']

for col in fin_cols:
    if col in df.columns:
        df[col] = safe_float(df[col])

print("Conversão numérica concluída.")

Conversão numérica concluída.


## 2. Refinamento de Rácios Financeiros

Complementamos os rácios existentes com métricas de rentabilidade e estrutura de capital.

In [3]:
# Liquidez (Já existe como 'Liquidity')
df['Current_Ratio'] = df['Liquidity']

# Solvabilidade / Leverage
df['Equity_to_Assets'] = df['Equity'] / df['TotalAssets'].replace(0, np.nan)
df['Debt_to_Assets'] = df['Indebtedness'] / 100.0  # Assumindo que Indebtedness está em %

# Rentabilidade
df['ROA_calc'] = df['NetProfit'] / df['TotalAssets'].replace(0, np.nan)
df['EBITDA_to_Assets'] = df['EBITDA'] / df['TotalAssets'].replace(0, np.nan)

# Eficiência
df['Asset_Turnover'] = df['Revenue'] / df['TotalAssets'].replace(0, np.nan)

print("Novos rácios calculados.")

Novos rácios calculados.


## 3. Tratamento de Outliers (Winsorization)

Removemos o ruído de valores extremos.

In [4]:
from scipy.stats.mstats import winsorize

ratio_cols = ['Current_Ratio', 'Equity_to_Assets', 'Debt_to_Assets', 'ROA_calc', 'EBITDA_to_Assets']

for col in ratio_cols:
    if col in df.columns:
        df[col] = winsorize(df[col].fillna(df[col].median()), limits=[0.01, 0.01])

print("Tratamento de outliers concluído.")

Tratamento de outliers concluído.


## 4. Variáveis de Governance — Análise de Missingness

A variável  apresenta uma taxa de valores omissos elevada (~63%). No contexto de PME portuguesas, este padrão é **potencialmente não aleatório (MNAR)**: empresas mais pequenas, de capital fechado ou menos transparentes tendem a não reportar a estrutura acionista no SABI.

A estratégia adoptada (implementada em ) é:
1. **Indicador de observabilidade**  (1 = reportado, 0 = omisso) — captura o sinal de transparência como feature preditiva independente.
2. **Imputação pela mediana** — valor neutro que preserva a distribuição dos que reportam sem introduzir viés direccional.
3. **Exclusão do lagging** — sendo estática (não varia no tempo), criar um lag seria redundante e propagaria NaNs.

A célula abaixo valida o mecanismo de missingness, comparando o perfil financeiro das empresas que reportam vs. as que não reportam.

In [5]:
ownership_obs = df['OwnershipConcentration'].notna().astype(int)
n_total = len(df['NIF Code'].unique())
n_observed = df.loc[df['OwnershipConcentration'].notna(), 'NIF Code'].nunique()

print(f'Observações totais : {len(df):,}')
print(f'Com OwnershipConcentration : {df["OwnershipConcentration"].notna().sum():,} ({df["OwnershipConcentration"].notna().mean():.1%})')
print(f'Sem OwnershipConcentration: {df["OwnershipConcentration"].isna().sum():,} ({df["OwnershipConcentration"].isna().mean():.1%})')
print(f'Empresas únicas com reporte: {n_observed:,} / {n_total:,} ({n_observed/n_total:.1%})')
print()

# Perfil financeiro: reporta vs. não reporta (teste MCAR informal)
df["_obs"] = ownership_obs
profile = df.groupby("_obs")[["TotalAssets", "Revenue", "FirmAge", "Employees"]].median()
profile.index = ["Não reporta (0)", "Reporta (1)"]
print("Mediana por grupo (Reporta vs. Não Reporta):")
print(profile.to_string())
df.drop(columns=["_obs"], inplace=True)

print()
print(f"Mediana OwnershipConcentration (base de imputação): {df['OwnershipConcentration'].median():.2f}")

Observações totais : 127,621
Com OwnershipConcentration : 40,052 (31.4%)
Sem OwnershipConcentration: 87,569 (68.6%)
Empresas únicas com reporte: 7,174 / 21,925 (32.7%)

Mediana por grupo (Reporta vs. Não Reporta):
                 TotalAssets  Revenue  FirmAge  Employees
Não reporta (0)         42.0     44.0      5.0        1.0
Reporta (1)             34.0     45.0      4.0        1.0

Mediana OwnershipConcentration (base de imputação): 100.00


## 5. Exportação do Dataset de Features

In [6]:
df.to_csv(os.path.join(DATA_INTERIM, "micro_features.csv"), index=False)
print(f"Sucesso! Dataset de features guardado em: {os.path.join(DATA_INTERIM, 'micro_features.csv')}")

Sucesso! Dataset de features guardado em: ../data/interim\micro_features.csv
